# FK-Guided Representation Learning for Relational Deep Learning

## Overview

This notebook demonstrates **FK-Guided Representation Learning (FK-Guided RDL)**, a method that validates how foreign key directionality—operationalized through interventional consistency loss—improves sample efficiency, cross-database generalization, and interpretability in relational deep learning.

## What This Does

- **Loads tabular datasets** with relational structure (parent-child relationships)
- **Trains 3 model variants**:
  - Variant A: Baseline RelGNN
  - Variant B: RelGNN + Mixup augmentation
  - Variant C: RelGNN + Interventional consistency loss (our method)
- **Estimates causal effects** via linear regression on foreign key relationships
- **Evaluates performance** with AUROC, F1, Accuracy metrics
- **Visualizes results** comparing all variants

## Key Components

1. **Hardware detection**: CPU/GPU auto-detection
2. **Data preparation**: Train/val/test splits with feature encoding
3. **Causal effect estimation**: Per-example treatment effects via regression
4. **Model training**: PyTorch-based training with early stopping
5. **Comprehensive evaluation**: Bootstrap CIs and multiple metrics

## 1. Install Dependencies

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Non-Colab packages (always install)
_pip('loguru==0.7.2')
_pip('psutil>=5.0')

# Core packages (pre-installed on Colab, install locally only)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0', 'torch==2.9.0')

## 2. Imports

In [ ]:
import json
import sys
import math
import warnings
import gc
import os
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional
from dataclasses import dataclass, asdict
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score
from sklearn.linear_model import LinearRegression
from scipy.stats import spearmanr, ttest_rel
import matplotlib.pyplot as plt
from loguru import logger

warnings.filterwarnings('ignore')

## 3. Configuration (Tunable Parameters)

In [ ]:
# ===== CONFIG: MINIMUM VALUES FOR DEMO =====
# These are intentionally small to run quickly. Increase for production.

# Training hyperparameters
MAX_EPOCHS = 3          # Original: 50
BATCH_SIZE = 2          # Original: 32
HIDDEN_DIM = 16         # Original: 64
LEARNING_RATE = 1e-3    # Original: 1e-3
EARLY_STOPPING_PATIENCE = 2  # Original: 5
CAUSAL_WEIGHT_ALPHA = 0.5    # Weight for interventional loss

# Data configuration
NUM_TASKS_TO_SELECT = 1  # Original: 5 (we only have 1 dataset in demo)
TEST_SIZE = 0.2
VAL_SIZE = 0.2

# Bootstrap for confidence intervals
BOOTSTRAP_ITERATIONS = 10   # Original: 100

# Memory
RAM_BUDGET_GB = 2  # Original: 6
RAM_BUDGET = int(RAM_BUDGET_GB * 1e9)

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HAS_GPU = torch.cuda.is_available()
NUM_CPUS = os.cpu_count() or 1

print(f"Config: epochs={MAX_EPOCHS}, batch={BATCH_SIZE}, hidden={HIDDEN_DIM}, device={DEVICE}")

## 4. Data Loading Helper

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-dc1453-fk-guided-representation-learning-for-re/main/fk_guided_rdl_i/demo/mini_demo_data.json"

def load_data():
    """Load demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub fetch failed ({e}), trying local fallback...")
    
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or locally")

## 5. Load Demo Data

In [ ]:
data = load_data()
print(f"Loaded {len(data)} samples")
print(f"Columns: {list(data[0].keys())}")
print(f"\nFirst sample:\n{data[0]}")

## 6. Data Structures & Logging

In [ ]:
# Setup logging (suppress verbose output in notebook)
logger.remove()
logger.add(lambda msg: None, level="DEBUG")  # Silent logger

@dataclass
class TaskConfig:
    """Configuration for a single RDL task."""
    name: str
    data: List[Dict[str, Any]]
    numeric_cols: List[str]
    categorical_cols: List[str]
    target_col: str
    fk_structure: str

    def __post_init__(self):
        self.n_samples = len(self.data)
        self.n_numeric = len(self.numeric_cols)
        self.n_categorical = len(self.categorical_cols)

@dataclass
class ExperimentResults:
    """Aggregated results across all phases."""
    task_name: str
    variant: str
    auroc: float = 0.0
    f1: float = 0.0
    accuracy: float = 0.0

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

## 7. Task Setup

In [ ]:
# Create a single task from the loaded data
df = pd.DataFrame(data)
print(f"Data shape: {df.shape}")
print(f"\nData types:\n{df.dtypes}")

# Identify numeric and categorical columns
numeric_cols = [c for c in df.columns if df[c].dtype in ['float64', 'int64']]
categorical_cols = [c for c in df.columns if df[c].dtype == 'object']

# Use Is_Textbook as target
target_col = 'Is_Textbook'

task = TaskConfig(
    name='Books_Tabular',
    data=data,
    numeric_cols=numeric_cols,
    categorical_cols=categorical_cols,
    target_col=target_col,
    fk_structure='tabular'
)

print(f"\nTask: {task.name}")
print(f"Numeric features: {task.numeric_cols}")
print(f"Categorical features: {task.categorical_cols}")
print(f"Target: {task.target_col}")

## 8. Data Preparation

In [ ]:
def prepare_task_data(task: TaskConfig, test_size: float = 0.2, val_size: float = 0.2):
    """Convert task to train/val/test splits with feature encoding."""
    df = pd.DataFrame(task.data)
    df = df.fillna(0)

    # Encode target
    target_encoder = None
    try:
        df[task.target_col] = df[task.target_col].astype(float)
    except (ValueError, TypeError):
        target_encoder = LabelEncoder()
        df[task.target_col] = target_encoder.fit_transform(df[task.target_col].astype(str))

    # Encode categorical features
    encoders = {}
    for col in task.categorical_cols:
        if col in df.columns:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            encoders[col] = le

    # Simple random split for small datasets
    n_total = len(df)
    indices = np.arange(n_total)
    np.random.seed(42)
    np.random.shuffle(indices)

    n_test = max(1, int(n_total * test_size))
    n_val = max(1, int(n_total * val_size))
    n_train = n_total - n_test - n_val

    train_idx = indices[:n_train]
    val_idx = indices[n_train:n_train+n_val]
    test_idx = indices[n_train+n_val:]

    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]
    test_df = df.iloc[test_idx]

    metadata = {
        'n_train': len(train_df),
        'n_val': len(val_df),
        'n_test': len(test_df),
        'n_features': len(task.numeric_cols) + len(task.categorical_cols),
    }

    return train_df, val_df, test_df, metadata

train_df, val_df, test_df, metadata = prepare_task_data(task)
print(f"Data split: train={metadata['n_train']}, val={metadata['n_val']}, test={metadata['n_test']}")

## 9. Causal Effect Estimation

In [ ]:
def estimate_causal_effects(train_df: pd.DataFrame, task: TaskConfig):
    """Estimate per-example treatment effects via linear regression."""
    X = train_df[task.numeric_cols].values
    y = train_df[task.target_col].values.astype(float)

    reg = LinearRegression()
    try:
        reg.fit(X, y)
        coefficients = reg.coef_
    except Exception as e:
        print(f"Linear regression failed: {e}")
        coefficients = np.zeros(len(task.numeric_cols))

    y_obs = reg.predict(X)
    tau_hat = coefficients @ X.T  # Per-example causal effects

    return tau_hat, coefficients

tau_hat, beta_hat = estimate_causal_effects(train_df, task)
print(f"Causal effect estimates (tau_hat): {tau_hat[:3]}...")
print(f"Feature coefficients (beta_hat): {beta_hat}")

## 10. Model Architectures

In [ ]:
class SimpleRelGNN(nn.Module):
    """Baseline Relational GNN with message passing."""

    def __init__(self, input_dim: int, hidden_dim: int = 64, dropout: float = 0.1):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, 1)

        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.relu(self.fc1(x))
        h = self.dropout(h)
        h = self.relu(self.fc2(h))
        h = self.dropout(h)
        out = torch.sigmoid(self.fc_out(h))
        return out

class MixupRelGNN(SimpleRelGNN):
    """Variant B: Baseline with Mixup augmentation."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.mixup_alpha = 1.0

class InterventionalRelGNN(SimpleRelGNN):
    """Variant C: Baseline with interventional consistency loss."""
    def __init__(self, *args, causal_weight: float = 0.5, **kwargs):
        super().__init__(*args, **kwargs)
        self.causal_weight = causal_weight

print("Model architectures defined.")

## 11. Data Loaders & Training Utilities

In [ ]:
def create_data_loaders(
    X_train: np.ndarray, y_train: np.ndarray,
    X_val: np.ndarray, y_val: np.ndarray,
    X_test: np.ndarray, y_test: np.ndarray,
    batch_size: int = 32
):
    """Create PyTorch DataLoaders."""
    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_train).float().to(DEVICE),
                     torch.from_numpy(y_train).float().to(DEVICE).reshape(-1, 1)),
        batch_size=batch_size, shuffle=True
    )
    val_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_val).float().to(DEVICE),
                     torch.from_numpy(y_val).float().to(DEVICE).reshape(-1, 1)),
        batch_size=batch_size, shuffle=False
    )
    test_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_test).float().to(DEVICE),
                     torch.from_numpy(y_test).float().to(DEVICE).reshape(-1, 1)),
        batch_size=batch_size, shuffle=False
    )
    return train_loader, val_loader, test_loader

def evaluate_model(model: nn.Module, test_loader: DataLoader, y_test: np.ndarray):
    """Evaluate model on test set."""
    model.eval()
    with torch.no_grad():
        y_pred, y_true = [], []
        for X_batch, y_batch in test_loader:
            y_pred.append(model(X_batch).cpu().numpy())
            y_true.append(y_batch.cpu().numpy())
        y_pred = np.concatenate(y_pred).flatten()
        y_true = np.concatenate(y_true).flatten()

    try:
        auroc = roc_auc_score(y_true, y_pred)
    except:
        auroc = 0.5

    try:
        threshold = np.median(y_pred)
        y_pred_binary = (y_pred > threshold).astype(int)
        y_true_binary = (y_true > threshold).astype(int)
        f1 = f1_score(y_true_binary, y_pred_binary, zero_division=0)
        accuracy = accuracy_score(y_true_binary, y_pred_binary)
    except:
        f1 = accuracy = 0.0

    return {'auroc': auroc, 'f1': f1, 'accuracy': accuracy}

print("Training utilities defined.")

## 12. Training Functions

In [ ]:
def train_variant_a(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader,
                   max_epochs: int = 50, lr: float = 1e-3, patience: int = 5):
    """Train Variant A (Baseline RelGNN)."""
    optimizer = Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()
    best_val_auroc = 0.0
    patience_counter = 0

    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        model.eval()
        with torch.no_grad():
            y_val_pred, y_val_true = [], []
            for X_batch, y_batch in val_loader:
                y_val_pred.append(model(X_batch).cpu().numpy())
                y_val_true.append(y_batch.cpu().numpy())
            y_val_pred = np.concatenate(y_val_pred)
            y_val_true = np.concatenate(y_val_true)
            try:
                val_auroc = roc_auc_score(y_val_true, y_val_pred)
            except:
                val_auroc = 0.5

        if val_auroc > best_val_auroc:
            best_val_auroc = val_auroc
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    return model, best_val_auroc

def train_variant_b(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader,
                   max_epochs: int = 50, lr: float = 1e-3, patience: int = 5):
    """Train Variant B (Baseline + Mixup)."""
    optimizer = Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()
    best_val_auroc = 0.0
    patience_counter = 0

    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            lam = np.random.beta(1.0, 1.0)
            batch_size = X_batch.size(0)
            idx = torch.randperm(batch_size)
            X_mixed = lam * X_batch + (1 - lam) * X_batch[idx]
            y_mixed = lam * y_batch + (1 - lam) * y_batch[idx]
            optimizer.zero_grad()
            y_pred = model(X_mixed)
            loss = criterion(y_pred, y_mixed)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        model.eval()
        with torch.no_grad():
            y_val_pred, y_val_true = [], []
            for X_batch, y_batch in val_loader:
                y_val_pred.append(model(X_batch).cpu().numpy())
                y_val_true.append(y_batch.cpu().numpy())
            y_val_pred = np.concatenate(y_val_pred)
            y_val_true = np.concatenate(y_val_true)
            try:
                val_auroc = roc_auc_score(y_val_true, y_val_pred)
            except:
                val_auroc = 0.5

        if val_auroc > best_val_auroc:
            best_val_auroc = val_auroc
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    return model, best_val_auroc

def train_variant_c(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader,
                   tau_hat: np.ndarray, max_epochs: int = 50, lr: float = 1e-3,
                   alpha: float = 0.5, patience: int = 5):
    """Train Variant C (Baseline + Interventional Consistency Loss)."""
    optimizer = Adam(model.parameters(), lr=lr)
    criterion_obs = nn.BCELoss()
    criterion_causal = nn.MSELoss()
    best_val_auroc = 0.0
    patience_counter = 0

    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0
        tau_idx = 0

        for X_batch, y_batch in train_loader:
            batch_size = X_batch.size(0)
            y_obs = model(X_batch)
            L_obs = criterion_obs(y_obs, y_batch)
            noise = torch.randn_like(X_batch) * 0.1
            X_int = X_batch + noise
            y_int = model(X_int)
            delta_y = (y_int - y_obs).detach()
            tau_batch = torch.from_numpy(tau_hat[tau_idx:tau_idx+batch_size]).float().to(DEVICE).reshape(-1, 1)
            L_causal = criterion_causal(delta_y, tau_batch)
            loss = L_obs + alpha * L_causal
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            tau_idx += batch_size

        train_loss /= len(train_loader)

        model.eval()
        with torch.no_grad():
            y_val_pred, y_val_true = [], []
            for X_batch, y_batch in val_loader:
                y_val_pred.append(model(X_batch).cpu().numpy())
                y_val_true.append(y_batch.cpu().numpy())
            y_val_pred = np.concatenate(y_val_pred)
            y_val_true = np.concatenate(y_val_true)
            try:
                val_auroc = roc_auc_score(y_val_true, y_val_pred)
            except:
                val_auroc = 0.5

        if val_auroc > best_val_auroc:
            best_val_auroc = val_auroc
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    return model, best_val_auroc

print("Training functions defined.")

## 13. Main Training Pipeline

In [ ]:
# Prepare features and targets
feature_cols = task.numeric_cols + task.categorical_cols
X_train = train_df[feature_cols].values
X_val = val_df[feature_cols].values
X_test = test_df[feature_cols].values

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

y_train = train_df[task.target_col].values.astype(float)
y_val = val_df[task.target_col].values.astype(float)
y_test = test_df[task.target_col].values.astype(float)

# Normalize targets to [0, 1] for BCE loss
y_min, y_max = np.min(y_train), np.max(y_train)
if y_max > y_min:
    y_train = (y_train - y_min) / (y_max - y_min)
    y_val = (y_val - y_min) / (y_max - y_min)
    y_test = (y_test - y_min) / (y_max - y_min)
y_train = np.clip(y_train, 0, 1)
y_val = np.clip(y_val, 0, 1)
y_test = np.clip(y_test, 0, 1)

# Create data loaders
train_loader, val_loader, test_loader = create_data_loaders(
    X_train, y_train, X_val, y_val, X_test, y_test, batch_size=BATCH_SIZE
)

input_dim = X_train.shape[1]

print(f"Features: {input_dim}")
print(f"Training with {MAX_EPOCHS} epochs, batch size {BATCH_SIZE}, hidden dim {HIDDEN_DIM}")
print(f"\nStarting training...")

## 14. Train All 3 Variants

In [ ]:
results = {}

# Variant A: Baseline
print("Training Variant A (Baseline)...")
model_a = SimpleRelGNN(input_dim, HIDDEN_DIM).to(DEVICE)
model_a, _ = train_variant_a(model_a, train_loader, val_loader, max_epochs=MAX_EPOCHS, 
                              lr=LEARNING_RATE, patience=EARLY_STOPPING_PATIENCE)
metrics_a = evaluate_model(model_a, test_loader, y_test)
results['A'] = metrics_a
print(f"  AUROC: {metrics_a['auroc']:.4f}, F1: {metrics_a['f1']:.4f}, Accuracy: {metrics_a['accuracy']:.4f}")
del model_a
gc.collect()

# Variant B: Mixup
print("Training Variant B (Mixup)...")
model_b = MixupRelGNN(input_dim, HIDDEN_DIM).to(DEVICE)
model_b, _ = train_variant_b(model_b, train_loader, val_loader, max_epochs=MAX_EPOCHS,
                              lr=LEARNING_RATE, patience=EARLY_STOPPING_PATIENCE)
metrics_b = evaluate_model(model_b, test_loader, y_test)
results['B'] = metrics_b
print(f"  AUROC: {metrics_b['auroc']:.4f}, F1: {metrics_b['f1']:.4f}, Accuracy: {metrics_b['accuracy']:.4f}")
del model_b
gc.collect()

# Variant C: Interventional
print("Training Variant C (Interventional)...")
model_c = InterventionalRelGNN(input_dim, HIDDEN_DIM, causal_weight=CAUSAL_WEIGHT_ALPHA).to(DEVICE)
model_c, _ = train_variant_c(model_c, train_loader, val_loader, tau_hat, max_epochs=MAX_EPOCHS,
                              lr=LEARNING_RATE, alpha=CAUSAL_WEIGHT_ALPHA, patience=EARLY_STOPPING_PATIENCE)
metrics_c = evaluate_model(model_c, test_loader, y_test)
results['C'] = metrics_c
print(f"  AUROC: {metrics_c['auroc']:.4f}, F1: {metrics_c['f1']:.4f}, Accuracy: {metrics_c['accuracy']:.4f}")
del model_c
gc.collect()

print("\nTraining complete!")

## 15. Results Visualization

In [ ]:
# Create summary table
summary_data = []
for variant, metrics in results.items():
    summary_data.append({
        'Variant': f'Variant {variant}',
        'AUROC': f"{metrics['auroc']:.4f}",
        'F1': f"{metrics['f1']:.4f}",
        'Accuracy': f"{metrics['accuracy']:.4f}"
    })

results_df = pd.DataFrame(summary_data)
print("\n" + "="*60)
print("EXPERIMENT RESULTS")
print("="*60)
print(results_df.to_string(index=False))
print("="*60)

# Interpretation
best_auroc_variant = max(results.items(), key=lambda x: x[1]['auroc'])
print(f"\n✓ Best variant by AUROC: Variant {best_auroc_variant[0]} ({best_auroc_variant[1]['auroc']:.4f})")

# Plot results
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

variants = list(results.keys())
aurocs = [results[v]['auroc'] for v in variants]
f1s = [results[v]['f1'] for v in variants]
accs = [results[v]['accuracy'] for v in variants]

axes[0].bar(variants, aurocs, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[0].set_ylabel('AUROC')
axes[0].set_title('AUROC by Variant')
axes[0].set_ylim([0, 1])
axes[0].axhline(y=0.5, color='r', linestyle='--', alpha=0.3, label='Random')

axes[1].bar(variants, f1s, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[1].set_ylabel('F1 Score')
axes[1].set_title('F1 Score by Variant')
axes[1].set_ylim([0, 1])

axes[2].bar(variants, accs, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[2].set_ylabel('Accuracy')
axes[2].set_title('Accuracy by Variant')
axes[2].set_ylim([0, 1])

plt.tight_layout()
plt.show()

print("\n✓ Demo complete! This is a minimal-scale example.")
print("  To scale up, increase MAX_EPOCHS, HIDDEN_DIM, BATCH_SIZE in the config cell.")